# Read from Bronze Table

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import StringType
from pyspark.sql.window import Window

In [0]:
raw_df = spark.table("workspace.bronze.crm_cust_info")
display(raw_df.limit(50))

# Data Transformations

In [0]:
rename_mapping = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date",
}
df = raw_df

# Trimming
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, F.trim(col(field.name)))

# Normalization of marital status, gndr
df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(col("cst_marital_status")) == "S", "Single")
        .when(F.upper(col("cst_marital_status")) == "M", "Married")
        .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(col("cst_gndr")) == "F", "Female")
        .when(F.upper(col("cst_gndr")) == "M", "Male")
        .otherwise("n/a")
    )
)

# Remove records with missing customer ID
df = df.filter(col("cst_id").isNotNull())

# Remove duplicate customer ID (keep most recent based on creation date)
window_spec = Window.partitionBy("cst_id").orderBy(col("cst_create_date").desc())
df = (
    df
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
)

# Column names are not friendly
for old_name, new_name in rename_mapping.items():
    df = df.withColumnRenamed(old_name, new_name)

display(df.limit(50))

# Write into Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")

# Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.crm_customers LIMIT 20;

In [0]:
%sql
SELECT customer_id FROM workspace.silver.crm_customers group by customer_id having count(*) > 1 LIMIT 20;